# 第 8 课：意图条件预测

目标：让检索同时考虑轨迹相似性与当前任务/机动意图，并识别标签泄漏风险。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 8.1 准备查询与记忆

核心源码：[retrieval.py](../src/memcast_uav/retrieval.py)  
文字讲解：[08_intent.md](../tutorial/08_intent.md)

In [ ]:
from memcast_uav.data import make_synthetic_flight, make_train_test_windows
from memcast_uav.features import extract_motion_features
from memcast_uav.memory import build_memory
from memcast_uav.retrieval import RetrievalConfig, retrieve

flight = make_synthetic_flight()
train, test = make_train_test_windows(flight, split_index=504)
memory = build_memory(train, limit=30)
query = test[3]
query_features = extract_motion_features(query.history, query.dt)
print("查询意图:", query.intent)

## 8.2 封装意图权重实验

In [ ]:
def retrieve_with_intent_weight(weight: float):
    return retrieve(
        query.history,
        query_features,
        query.intent,
        memory,
        RetrievalConfig(
            alpha=0.5,
            gamma=120.0,
            top_k=3,
            intent_weight=weight,
        ),
    )

without_intent = retrieve_with_intent_weight(0.0)
with_intent = retrieve_with_intent_weight(0.6)

## 8.3 比较 Top-3

In [ ]:
def summarize(results):
    return [
        {
            "id": item.entry.entry_id,
            "intent": item.entry.intent,
            "match": item.intent_match,
            "score": round(item.final_score, 4),
        }
        for item in results
    ]

print("不使用意图:")
for item in summarize(without_intent):
    print(item)

print("\n提高意图权重:")
for item in summarize(with_intent):
    print(item)

## 8.4 意图来源与泄漏

- 任务规划：巡检、配送、返航，通常在预测时可见；
- 控制指令：左转、爬升、悬停，通常在预测时可见；
- 在线识别：只能根据已经观测到的历史估计；
- **禁止**读取真实未来轨迹后再反推当前意图。

## 8.5 分块练习：设计层级意图

In [ ]:
hierarchical_intent = {
    "mission": "inspection",
    "maneuver": "turn_right",
}
mission_weight = 0.2
maneuver_weight = 0.6
print(hierarchical_intent)
print("TODO：设计 mission_match 和 maneuver_match 的组合分数。")

## 8.6 本课验收

In [ ]:
assert len(without_intent) == 3
assert len(with_intent) == 3

import subprocess

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_retrieval.py",
        "tests/test_pipeline.py",
        "-q",
    ],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

## 下一阶段

先阅读 [真实无人机数据迁移指南](../docs/03_uav_dataset_adapter.md)，把 DJI Matrice 100
或 NeuroBEM/UZH-FPV 转换为统一窗口，再考虑接入 Qwen。

[← 第 7 课](07_end_to_end.ipynb) · [教程目录](README.md) ·
[真实无人机数据迁移 →](../docs/03_uav_dataset_adapter.md)